
# Graph Fusion GNN Experiment

This notebook trains a **Graph Neural Network (GraphFusionNet)** using static and dynamic data features and evaluates its performance using t-SNE visualization.

---
### **Pipeline Overview**
1. Configuration  
2. Load dataset  
3. Label nodes  
4. Extract feature groups  
5. Normalize features  
6. Convert to tensors  
7. Create masks (train/val/test)  
8. Train model  
9. Evaluate and visualize embeddings


**GraphFusionNet-1 Architecture:**  
GraphFusionNet-1 is a dual-branch graph neural network designed to fuse node features from two distinct graphs sharing the same topology. Each input (`x1`, `x2`) passes through its own graph convolution layer (`gcn1_1`, `gcn1_2`) to produce hidden representations (`h1`, `h2`). These are combined using a learnable fusion weight α via a weighted sum \( h = αh_1 + (1 - α)h_2 \). The fused embedding `h` then passes through another pair of graph convolution layers (`gcn2_1`, `gcn2_2`), followed by a second fusion stage using the same α parameter to yield the final output. This design enables adaptive feature blending from multiple graph sources, enhancing joint representation learning.


**GraphFusionNet-1 Summary:**

- Designed for fusing static and dynamic graph data.  
- Inputs: node features `x1` (static), `x2` (dynamic), and shared `edge_index`.  
- Two parallel GCN layers extract features from both data sources. 
- Each branch has its own graph convolution layer (`gcn1_1` and `gcn1_2`).  
- Outputs from both branches (`h1`, `h2`) are combined using a learnable weight **α**.  
- Fusion equation:
  \( h = αh_1 + (1 - α)h_2 \).  
- A second GCN layer refines the fused representation into final output.  
- Returns either the fused embedding (`h`) or final output (`out`). 


In [2]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, SAGEConv, GATConv, GraphConv
from torch_geometric.data import Data
import optuna
import pandas as pd
import numpy as np
import torch.nn as nn
import os
import matplotlib.pyplot as plt
import re
from sklearn.preprocessing import MinMaxScaler
import seaborn as sns
from sklearn.manifold import TSNE
from torch_geometric.loader import DataLoader



from helper_funcs import *
from plots import *
from models import *
from model_helper_funcs import *
from model_fusion_scratch import *

c:\Users\waqar\AppData\Local\pypoetry\Cache\virtualenvs\data-fusion-r9UYRqEp-py3.12\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Configuration


In [ ]:
seed = 42
set_seed(seed)

num_epochs = 200

data_set = "real"         # "real" | "artificial"
type_graph_conv = 'StaticDynamicGNN_adv_3'  # StaticDynamicGNN_hidden "SAGE" | "GCN" | "GAT" | "FWAGCN" | "EFWAGCN" | "GraphConv" |DualGNN | StaticDynamicGNN_adv
architecture = "2_layers"

# if type_graph_conv == 'SAGE':
#     graph_conv = SAGEConv
# elif type_graph_conv == "GCN":
#     graph_conv = GCNConv
# elif type_graph_conv == "GAT":
#     graph_conv = GATConv
# elif type_graph_conv == "FWAGCN":
#     graph_conv = FeatureWiseGraphConv
# elif type_graph_conv == "EFWAGCN":
#     graph_conv = EdgeFeatureAttentionGCN
# elif type_graph_conv == "GraphConv":
#     graph_conv = GraphConv
# elif type_graph_conv == "DualGNN":
#     graph_conv = DualGNN
    
    
model_used = "M33_Hierarchical_Fusion_Complete_Explainability"     

output_folder = f"results/{model_used}/{type_graph_conv}/{architecture}"
os.makedirs(output_folder, exist_ok=True)

# Load dataset

In [4]:
if data_set == "artificial":
    print("Using artificial dataset")
    name_file_dataset = os.path.join("datasets", "artificial_data_12_clusters_full.csv")
else:
    print("Using real dataset")
    name_file_dataset = "datasets/datasubset_nodes_waqar.csv"

df_nodes = pd.read_csv(name_file_dataset)
df_edges = pd.read_csv("datasets/aristas_subgrafoSPdaily.csv")
edge_index = map_edges_new_index(df_nodes, df_edges)

Using real dataset
tensor([[    573641,     573641,     573643,  ..., 8751085661, 8753316632,
         8795904828],
        [ 465879071,  292424978,  292424978,  ..., 8751085667, 8753316633,
         8795904831]])


d:\data_fusion\helper_funcs.py:50: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  edges_mapeados = df_edges.applymap(lambda x: mapping_dict.get(x, x)).values


edge_index_mapeado ------->  tensor([[    0,     0,     1,  ..., 14575, 14578, 14596],
        [ 5698,  1381,  1381,  ..., 14576, 14579, 14597]])


In [5]:
df_nodes.shape

(14640, 159)

# Label nodes

In [6]:
df_nodes_label = label_crime_nodes(df_nodes, threshold=6)
crime_label_col = "crime_label"
num_nodes = df_nodes_label.shape[0]

In [7]:
# plot_crime_vs_nocrime_ratio(df_nodes_label)

# Extract feature groups

In [8]:
df_nodes_label.columns

Index(['Nodo', '2006.01', '2006.02', '2006.03', '2006.04', '2006.05',
       '2006.06', '2006.07', '2006.08', '2006.09',
       ...
       'maiores_de_65_anos_taxa', 'Pontos_de_onibus', 'Estacao_de_metro',
       'Estacao_de_trem', 'Terminal_de_onibus', 'Favela_proxima', 'lat',
       'long', 'total_crimes', 'crime_label'],
      dtype='object', length=161)

In [9]:
dynamic_cols = [col for col in df_nodes_label.columns if re.match(r'^\d{4}\.\d{2}$', col)]
dynamic_dt = df_nodes_label[dynamic_cols]

cols_to_drop = dynamic_cols + ['Nodo', 'Pontos_de_onibus', 'lat', 'long', 'total_crimes', 'crime_label']
static_dt = df_nodes_label.drop(columns=cols_to_drop)

In [10]:
crime_label_col = df_nodes_label['crime_label']
crime_label_col

0        1
1        1
2        1
3        1
4        0
        ..
14635    0
14636    0
14637    0
14638    0
14639    0
Name: crime_label, Length: 14640, dtype: int64

In [11]:
static_dt.shape

(14640, 11)

# Normalize features


# Smooth dynamic data

In [ ]:
dynamic_dt_smooth = dynamic_dt.apply(lambda row: row.ewm(alpha=0.1).mean(), axis=1)
dynamic_dt_smooth_log = np.log1p(np.array(dynamic_dt_smooth))

# global_min = dynamic_dt_smooth_log.min().min()
# global_max = dynamic_dt_smooth_log.max().max()
# dynamic_dt_norm = pd.DataFrame((dynamic_dt_smooth_log - global_min) / (global_max - global_min))

dynamic_dt_smooth_log = pd.DataFrame(dynamic_dt_smooth_log)
static_dt = pd.DataFrame(static_dt)

# scaler = MinMaxScaler()

# static_dt_norm = pd.DataFrame(scaler.fit_transform(static_dt))

# Convert to tensors

In [ ]:
dynamic_tensor = torch.tensor(dynamic_dt_smooth_log.values, dtype=torch.float32)
dynamic_tensor = torch.nan_to_num(dynamic_tensor, nan=0.0)

static_tensor = torch.tensor(static_dt.values, dtype=torch.float32)
static_tensor = torch.nan_to_num(static_tensor, nan=0.0)

In [15]:
print(dynamic_tensor.shape)
print(static_tensor.shape)

torch.Size([14640, 144])
torch.Size([14640, 11])


# Sliding Window approach

In [ ]:
# # Time split
TRAIN_END = 100
T = dynamic_tensor.size(1)
W = 5

# train_dataset = CrimeWindowPyGDataset(
# # train_dataset = CrimeWindowDataset(
#     static_tensor, dynamic_tensor, edge_index,
#     window_size=W,
#     start_t=0,
#     end_t=TRAIN_END
# )

# test_dataset = CrimeWindowPyGDataset(
# # test_dataset = CrimeWindowDataset(
#     static_tensor, dynamic_tensor, edge_index,
#     window_size=W,
#     start_t=TRAIN_END,
#     end_t=T
# )

# Inductive Learning

In [ ]:
import torch

def filter_and_reindex(data, allowed_nodes):

    # ---- Step 1: mask nodes ----
    mask = torch.isin(data.node_ids, allowed_nodes)

    if mask.sum() == 0:
        return None

    # ---- Step 2: filter node features ----
    new_x_static = data.x_static[mask]
    new_x_dynamic = data.x_dynamic[mask]
    new_y = data.y[mask]

    # IMPORTANT: keep original node ids temporarily
    old_node_ids = data.node_ids[mask]

    # ---- Step 3: create mapping old → new index ----
    # Example: {5:0, 10:1, 20:2}
    unique_nodes = old_node_ids
    mapping = {int(old_id): i for i, old_id in enumerate(unique_nodes)}

    # ---- Step 4: filter edges ----
    src, dst = data.edge_index

    edge_mask = torch.isin(src, old_node_ids) & torch.isin(dst, old_node_ids)

    src = src[edge_mask]
    dst = dst[edge_mask]

    # ---- Step 5: reindex edges ----
    new_src = torch.tensor([mapping[int(s)] for s in src])
    new_dst = torch.tensor([mapping[int(d)] for d in dst])

    new_edge_index = torch.stack([new_src, new_dst], dim=0)

    # ---- Step 6: create new node ids (0...N-1) ----
    new_node_ids = torch.arange(len(unique_nodes))

    # ---- Step 7: update time_id ----
    new_time_id = data.time_id[mask]

    # ---- Step 8: return new Data ----
    new_data = data.clone()

    new_data.x_static = new_x_static
    new_data.x_dynamic = new_x_dynamic
    new_data.y = new_y
    new_data.edge_index = new_edge_index
    new_data.node_ids = new_node_ids   # ✅ reindexed
    new_data.time_id = new_time_id
    new_data.num_nodes = len(new_node_ids)

    return new_data

from torch.utils.data import Dataset

class InductiveWrapperDataset(Dataset):
    def __init__(self, base_dataset, allowed_nodes):
        self.base_dataset = base_dataset
        self.allowed_nodes = allowed_nodes

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):

        data = self.base_dataset[idx]

        new_data = filter_and_reindex(data, self.allowed_nodes)

        # Handle rare empty case
        if new_data is None:
            return self.__getitem__((idx + 1) % len(self))

        return new_data
    
    import numpy as np

def split_nodes(num_nodes, train_ratio=0.8, seed=42):
    np.random.seed(seed)

    nodes = np.arange(num_nodes)
    np.random.shuffle(nodes)

    split = int(train_ratio * num_nodes)

    train_nodes = torch.tensor(nodes[:split])
    test_nodes = torch.tensor(nodes[split:])

    return train_nodes, test_nodes

## Split nodes

In [ ]:
train_nodes, test_nodes = split_nodes(static_tensor.shape[0])

## Compute normalization ONLY from TRAIN

Dynamic normalization

In [ ]:
train_dynamic = dynamic_tensor[train_nodes][:, :TRAIN_END]

min_val = train_dynamic.min()
max_val = train_dynamic.max()

dynamic_tensor_norm = (dynamic_tensor - min_val) / (max_val - min_val + 1e-8)

Data(edge_index=[2, 55234], y=[14640], x_static=[14640, 11], x_dynamic=[14640, 5], node_ids=[14640], time_id=[14640], num_nodes=14640)

In [ ]:
dynamic_tensor.max()

Static normalization

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

train_static = static_tensor[train_nodes]

scaler.fit(train_static)

static_tensor_norm = torch.tensor(
    scaler.transform(static_tensor),
    dtype=torch.float32
)

Temporal Split

In [ ]:
train_dataset = CrimeWindowPyGDataset(
    static_tensor_norm,
    dynamic_tensor_norm,
    edge_index,
    window_size=W,
    start_t=0,
    end_t=TRAIN_END
)

test_dataset = CrimeWindowPyGDataset(
    static_tensor_norm,
    dynamic_tensor_norm,
    edge_index,
    window_size=W,
    start_t=TRAIN_END,
    end_t=T
)

# Inductive Split on nodes

In [ ]:

# train_dataset = InductiveWrapperDataset(train_dataset, train_nodes)
# test_dataset = InductiveWrapperDataset(test_dataset, test_nodes)

tensor([[    0,     0,     0,  ..., 14639, 14639, 14639],
        [    0,  1381,  5698,  ...,   134, 12052, 14639]])

In [ ]:
sample = train_dataset[0]

print(sample.x_static.shape)
print(sample.edge_index.max(), sample.num_nodes)

In [ ]:
sample = test_dataset[0]

print(sample.x_dynamic.shape)
print(sample.edge_index.max(), sample.num_nodes)

# Hierarchical static–dynamic graph fusion

This section replaces only the model and post-model workflow. All dataset loading, feature extraction, smoothing, normalization, tensor conversion, temporal splitting and sliding-window construction above remain unchanged.

## Architecture

The hierarchy has three representation levels:

1. **Modality-specific graph encoding**

$$z_s=mathrm{ReLU}(mathrm{GAT}_s(x_s,E)),qquad
z_d=mathrm{ReLU}(mathrm{GAT}_d(x_d,E)).$$

2. **Cross-modality fusion**

$$h_f=mathrm{ReLU}(W_f[z_sVert z_d]+b_f).$$

3. **Joint graph refinement and prediction**

$$h_j=mathrm{ReLU}(mathrm{GAT}_{joint}(h_f,E)),qquad
hat y=mathrm{MLP}(h_j).$$

Late fusion predicts immediately after concatenating the independent branches. Hierarchical fusion instead learns a joint representation and performs another graph-propagation stage after fusion.

## Hierarchical fusion architecture code

In [ ]:
import copy
from torch.utils.data import Subset
from torch_geometric.nn import GraphConv


class StaticHierarchicalEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.conv = GraphConv(in_dim, hidden_dim)

    def forward(self, x_static, edge_index):
        return F.relu(self.conv(x_static, edge_index))


class DynamicHierarchicalEncoder(nn.Module):
    def __init__(self, window_size, hidden_dim):
        super().__init__()
        self.conv = GraphConv(window_size, hidden_dim)

    def forward(self, x_dynamic, edge_index):
        return F.relu(self.conv(x_dynamic, edge_index))


class HierarchicalStaticDynamicGNN(nn.Module):
    def __init__(self, in_static, window_size, hidden_dim=64, dropout=0.2):
        super().__init__()
        self.static_encoder = StaticHierarchicalEncoder(in_static, hidden_dim)
        self.dynamic_encoder = DynamicHierarchicalEncoder(window_size, hidden_dim)

        # Level 2: fuse the two graph-encoded modalities.
        self.fusion_mlp = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        # Level 3: jointly propagate the already-fused representation.
        self.joint_conv = GraphConv(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.predictor = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward_from_encoded(
        self, z_static, z_dynamic, edge_index,
        return_details=False, bypass_joint_graph=False
    ):
        concatenated = torch.cat([z_static, z_dynamic], dim=-1)
        h_fused = self.fusion_mlp(concatenated)
        if bypass_joint_graph:
            h_joint = h_fused
        else:
            h_joint = F.relu(self.joint_conv(h_fused, edge_index))
        h_joint = self.dropout(h_joint)
        prediction = self.predictor(h_joint).squeeze(-1)
        if return_details:
            return prediction, h_joint, {
                "z_static": z_static,
                "z_dynamic": z_dynamic,
                "h_fused": h_fused,
                "h_joint": h_joint,
            }
        return prediction, h_joint

    def forward(
        self, x_static, x_dynamic, edge_index,
        return_details=False, bypass_joint_graph=False
    ):
        z_static = self.static_encoder(x_static, edge_index)
        z_dynamic = self.dynamic_encoder(x_dynamic, edge_index)
        return self.forward_from_encoded(
            z_static, z_dynamic, edge_index,
            return_details=return_details,
            bypass_joint_graph=bypass_joint_graph,
        )


class CrimeForecastModel_hierarchical_fusion(nn.Module):
    def __init__(self, in_static, window_size, hidden_dim=64, dropout=0.2):
        super().__init__()
        self.hierarchical_fusion = HierarchicalStaticDynamicGNN(
            in_static=in_static,
            window_size=window_size,
            hidden_dim=hidden_dim,
            dropout=dropout,
        )

    def forward(self, x_static, x_dynamic, edge_index,
                return_details=False, bypass_joint_graph=False):
        return self.hierarchical_fusion(
            x_static, x_dynamic, edge_index,
            return_details=return_details,
            bypass_joint_graph=bypass_joint_graph,
        )


## Model initialization

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
S = static_tensor.size(1)
hidden_dim = 64

model = CrimeForecastModel_hierarchical_fusion(
    in_static=S,
    window_size=W,
    hidden_dim=hidden_dim,
    dropout=0.2,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.MSELoss()

model_used = "M8_hierarchical_fusion"
type_graph_conv = "GAT"
architecture = "separate_encoders_fusion_joint_graph"
output_folder = f"results/{model_used}/{type_graph_conv}/{architecture}"
os.makedirs(output_folder, exist_ok=True)

print(model)
print("Device:", device)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))


## Chronological validation split

The latest 20% of the existing training windows are used for validation. The test period remains untouched until final evaluation.

In [ ]:
validation_fraction = 0.20
num_development_windows = len(train_dataset)
num_validation_windows = max(1, int(np.ceil(validation_fraction * num_development_windows)))
validation_start = num_development_windows - num_validation_windows

fit_dataset = Subset(train_dataset, range(0, validation_start))
validation_dataset = Subset(train_dataset, range(validation_start, num_development_windows))

train_loader = DataLoader(fit_dataset, batch_size=1, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=1, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

print("Fit windows:", len(fit_dataset))
print("Validation windows:", len(validation_dataset))
print("Untouched test windows:", len(test_dataset))


## Leakage-free training and checkpoint selection

In [ ]:
def train_hierarchical_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_cases = 0
    static_norm_sum = 0.0
    dynamic_norm_sum = 0.0

    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        prediction, _, details = model(
            batch.x_static, batch.x_dynamic, batch.edge_index,
            return_details=True,
        )
        target = batch.y.float().view_as(prediction)
        loss = criterion(prediction, target)
        loss.backward()
        optimizer.step()

        n = target.numel()
        total_loss += loss.item() * n
        total_cases += n
        static_norm_sum += torch.linalg.vector_norm(
            details["z_static"].detach(), dim=1
        ).sum().item()
        dynamic_norm_sum += torch.linalg.vector_norm(
            details["z_dynamic"].detach(), dim=1
        ).sum().item()

    return (
        total_loss / max(total_cases, 1),
        static_norm_sum / max(total_cases, 1),
        dynamic_norm_sum / max(total_cases, 1),
    )


@torch.no_grad()
def evaluate_hierarchical_loss(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_cases = 0
    for batch in loader:
        batch = batch.to(device)
        prediction, _ = model(batch.x_static, batch.x_dynamic, batch.edge_index)
        target = batch.y.float().view_as(prediction)
        loss = criterion(prediction, target)
        n = target.numel()
        total_loss += loss.item() * n
        total_cases += n
    return total_loss / max(total_cases, 1)


max_epochs = 250
patience = 15
min_delta = 1e-6

train_loss_history = []
validation_loss_history = []
static_encoder_norm_history = []
dynamic_encoder_norm_history = []

best_validation_loss = float("inf")
best_epoch = 0
best_model_state = None
epochs_without_improvement = 0

for epoch in range(max_epochs):
    train_loss, static_norm, dynamic_norm = train_hierarchical_epoch(
        model, train_loader, optimizer, criterion, device
    )
    validation_loss = evaluate_hierarchical_loss(
        model, validation_loader, criterion, device
    )

    train_loss_history.append(train_loss)
    validation_loss_history.append(validation_loss)
    static_encoder_norm_history.append(static_norm)
    dynamic_encoder_norm_history.append(dynamic_norm)

    if validation_loss < best_validation_loss - min_delta:
        best_validation_loss = validation_loss
        best_epoch = epoch + 1
        best_model_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 0 or (epoch + 1) % 10 == 0 or epoch + 1 == max_epochs:
        print(
            f"Epoch {epoch+1:03d}/{max_epochs} | "
            f"Train MSE: {train_loss:.8f} | "
            f"Validation MSE: {validation_loss:.8f} | "
            f"Best: {best_validation_loss:.8f} (epoch {best_epoch}) | "
            f"Patience: {epochs_without_improvement}/{patience}"
        )

    if epochs_without_improvement >= patience:
        print(f"Early stopping at epoch {epoch+1}; best epoch was {best_epoch}.")
        break

if best_model_state is None:
    raise RuntimeError("No validation checkpoint was created.")

model.load_state_dict(best_model_state)
num_trained_epochs = len(train_loss_history)
print(f"Restored epoch {best_epoch} with validation MSE {best_validation_loss:.8f}.")


## Training diagnostics

In [ ]:
epochs = np.arange(1, len(train_loss_history)+1)
fig, axes = plt.subplots(1,2,figsize=(12,4.5))
axes[0].plot(epochs,train_loss_history,label="Training MSE")
axes[0].plot(epochs,validation_loss_history,label="Validation MSE")
axes[0].axvline(best_epoch,color="red",linestyle="--",label=f"Best epoch: {best_epoch}")
axes[0].set(title="Hierarchical-fusion model selection",xlabel="Epoch",ylabel="MSE"); axes[0].legend(); axes[0].grid(alpha=.2)

static_norm=np.asarray(static_encoder_norm_history); dynamic_norm=np.asarray(dynamic_encoder_norm_history)
total=static_norm+dynamic_norm+1e-12
axes[1].plot(epochs,static_norm/total,label="Static encoder norm share",color="tab:orange")
axes[1].plot(epochs,dynamic_norm/total,label="Dynamic encoder norm share",color="tab:green",linestyle="--")
axes[1].set(title="Encoder activation magnitude during fitting",xlabel="Epoch",ylabel="Norm share",ylim=(0,1)); axes[1].legend(); axes[1].grid(alpha=.2)
plt.tight_layout(); plt.savefig(os.path.join(output_folder,"hierarchical_training_diagnostics.png"),dpi=300,bbox_inches="tight"); plt.show()


# Complete hierarchical-fusion analysis suite

## Applicability map

| Analysis | Hierarchical-fusion status |
|---|---|
| Prediction metrics and residuals | Applicable |
| Raw gate distribution/entropy/saturation | **Not applicable: no gate exists** |
| Feature-wise gate specialization | **Not applicable** |
| Encoder-channel activation analysis | Applicable analogue |
| Gate–crime association | **Not applicable** |
| Occlusion-based branch-share association | Applicable analogue |
| Self-versus-neighbor gates | **Not applicable** |
| Branch removal and shuffling | Applicable |
| Input permutation importance | Applicable |
| Hierarchy-level intervention | Applicable |
| Graph reliance | Applicable |
| Spatial/temporal stability | Applicable |
| Time-cluster bootstrap | Applicable |

Because nonlinear fusion and joint graph propagation introduce interactions, modality contributions are not exactly additive. The analysis uses branch occlusion and reports the interaction term rather than pretending that a probability gate exists.

## A. Rich test collection and nonlinear branch-occlusion contribution

In [ ]:
import warnings
try:
    from scipy.stats import pearsonr, spearmanr
    SCIPY_AVAILABLE = True
except ImportError:
    SCIPY_AVAILABLE = False
    warnings.warn("SciPy unavailable; p-values will be skipped.")


def _expand_hier_metadata(value,n,fallback,device):
    if value is None: value=torch.as_tensor(fallback,device=device)
    value=value.reshape(-1)
    if value.numel()==1 and n>1: value=value.repeat(n)
    if value.numel()!=n: raise ValueError(f"Metadata length {value.numel()} does not match {n} cases")
    return value


@torch.no_grad()
def collect_hierarchical_explainability(model,loader,device):
    model.eval(); frames=[]; zs_all=[]; zd_all=[]; hf_all=[]; hj_all=[]; predictions_per_time=[]; targets_per_time=[]
    core=model.hierarchical_fusion
    for step,batch in enumerate(loader):
        batch=batch.to(device)
        prediction,h_joint,details=model(batch.x_static,batch.x_dynamic,batch.edge_index,return_details=True)
        zs=details["z_static"]; zd=details["z_dynamic"]
        pred_no_static,_,_=core.forward_from_encoded(torch.zeros_like(zs),zd,batch.edge_index,return_details=True)
        pred_no_dynamic,_,_=core.forward_from_encoded(zs,torch.zeros_like(zd),batch.edge_index,return_details=True)
        pred_neither,_,_=core.forward_from_encoded(torch.zeros_like(zs),torch.zeros_like(zd),batch.edge_index,return_details=True)
        target=batch.y.float().view_as(prediction); n=target.numel()
        static_effect=(prediction-pred_no_static).abs(); dynamic_effect=(prediction-pred_no_dynamic).abs()
        static_share=static_effect/(static_effect+dynamic_effect+1e-12)
        interaction=prediction-pred_no_static-pred_no_dynamic+pred_neither
        node_ids=_expand_hier_metadata(getattr(batch,"node_ids",None),n,torch.arange(n,device=device),device)
        time_ids=_expand_hier_metadata(getattr(batch,"time_id",None),n,step,device)
        frames.append(pd.DataFrame({
            "node_id":node_ids.cpu().numpy(),"time_id":time_ids.cpu().numpy(),
            "input_crime":batch.x_dynamic.mean(dim=1).cpu().numpy(),"target_crime":target.cpu().numpy(),
            "prediction":prediction.cpu().numpy(),"residual":(target-prediction).cpu().numpy(),"absolute_error":(target-prediction).abs().cpu().numpy(),
            "static_occlusion_effect":static_effect.cpu().numpy(),"dynamic_occlusion_effect":dynamic_effect.cpu().numpy(),
            "static_effective_share":static_share.cpu().numpy(),"dynamic_effective_share":(1-static_share).cpu().numpy(),
            "interaction_output":interaction.cpu().numpy(),
            "static_embedding_norm":torch.linalg.vector_norm(zs,dim=1).cpu().numpy(),"dynamic_embedding_norm":torch.linalg.vector_norm(zd,dim=1).cpu().numpy(),
            "fused_embedding_norm":torch.linalg.vector_norm(details["h_fused"],dim=1).cpu().numpy(),"joint_embedding_norm":torch.linalg.vector_norm(h_joint,dim=1).cpu().numpy(),
        }))
        zs_all.append(zs.cpu()); zd_all.append(zd.cpu()); hf_all.append(details["h_fused"].cpu()); hj_all.append(h_joint.cpu())
        predictions_per_time.append(prediction.cpu().numpy()); targets_per_time.append(target.cpu().numpy())
    frame=pd.concat(frames,ignore_index=True); error=frame.prediction.to_numpy()-frame.target_crime.to_numpy(); target=frame.target_crime.to_numpy(); mse=float(np.mean(error**2)); denom=float(np.sum((target-target.mean())**2))
    metrics={"MSE":mse,"RMSE":float(np.sqrt(mse)),"MAE":float(np.mean(np.abs(error))),"R2":float(1-np.sum(error**2)/denom) if denom>0 else np.nan}
    details={"frame":frame,"z_static":torch.cat(zs_all).numpy(),"z_dynamic":torch.cat(zd_all).numpy(),"h_fused":torch.cat(hf_all).numpy(),"h_joint":torch.cat(hj_all).numpy(),"predictions_per_time":predictions_per_time,"targets_per_time":targets_per_time}
    return metrics,details


hierarchical_metrics,hierarchical_details=collect_hierarchical_explainability(model,test_loader,device)
hierarchical_frame=hierarchical_details["frame"]
display(pd.DataFrame([hierarchical_metrics])); display(hierarchical_frame.head())
hierarchical_frame.to_csv(os.path.join(output_folder,"hierarchical_rich_explainability.csv"),index=False)


## B. Gate health and self-versus-neighbor gates — not applicable

This hierarchy uses concatenation, an MLP and joint GAT refinement. It contains no sigmoid or softmax modality gate. Therefore, gate entropy, gate saturation, gate shuffling, feature-wise gate curves and separate self/neighbor modality gates are undefined. Encoder norms and occlusion shares below are diagnostics, not gates.

## C. Global branch strength and nonlinear interaction

In [ ]:
hierarchical_global_contribution=pd.DataFrame([{
    "mean_static_occlusion_effect":hierarchical_frame.static_occlusion_effect.mean(),
    "mean_dynamic_occlusion_effect":hierarchical_frame.dynamic_occlusion_effect.mean(),
    "mean_static_effective_share":hierarchical_frame.static_effective_share.mean(),
    "mean_dynamic_effective_share":hierarchical_frame.dynamic_effective_share.mean(),
    "mean_absolute_interaction":hierarchical_frame.interaction_output.abs().mean(),
    "mean_static_embedding_norm":hierarchical_frame.static_embedding_norm.mean(),
    "mean_dynamic_embedding_norm":hierarchical_frame.dynamic_embedding_norm.mean(),
}])
display(hierarchical_global_contribution.round(6))
fig,axes=plt.subplots(1,3,figsize=(16,4.5))
axes[0].hist(hierarchical_frame.static_effective_share,bins=35,color="tab:orange",alpha=.8); axes[0].set(title="Occlusion-based static share",xlabel="Static effective share",ylabel="Cases")
axes[1].hist(hierarchical_frame.interaction_output,bins=35,color="tab:purple",alpha=.8); axes[1].axvline(0,color="black",linestyle="--"); axes[1].set(title="Non-additive interaction output",xlabel="Interaction")
axes[2].bar(["Static","Dynamic"],[hierarchical_frame.static_occlusion_effect.mean(),hierarchical_frame.dynamic_occlusion_effect.mean()],color=["tab:orange","tab:green"]); axes[2].set(title="Mean branch-removal effect",ylabel="Mean absolute prediction change")
for ax in axes: ax.grid(alpha=.2)
plt.tight_layout(); plt.savefig(os.path.join(output_folder,"hierarchical_global_contribution.png"),dpi=300,bbox_inches="tight"); plt.show()
hierarchical_global_contribution.to_csv(os.path.join(output_folder,"hierarchical_global_contribution.csv"),index=False)


## D. Encoder-channel specialization

There are no channel-wise gates. This analysis instead measures the mean activation and variability of each modality-specific encoder channel before fusion.

In [ ]:
zs=hierarchical_details["z_static"]; zd=hierarchical_details["z_dynamic"]; dims=np.arange(zs.shape[1])
hierarchical_channel_table=pd.DataFrame({"channel":dims,"static_mean_activation":zs.mean(axis=0),"static_std_activation":zs.std(axis=0),"dynamic_mean_activation":zd.mean(axis=0),"dynamic_std_activation":zd.std(axis=0)})
fig,axes=plt.subplots(1,2,figsize=(14,4.8))
axes[0].errorbar(dims,zs.mean(axis=0),yerr=zs.std(axis=0),marker="o",markersize=3,label="Static",color="tab:orange")
axes[0].set(title="Static encoder channels",xlabel="Channel",ylabel="Activation mean ± SD")
axes[1].errorbar(dims,zd.mean(axis=0),yerr=zd.std(axis=0),marker="o",markersize=3,label="Dynamic",color="tab:green")
axes[1].set(title="Dynamic encoder channels",xlabel="Channel",ylabel="Activation mean ± SD")
for ax in axes: ax.grid(alpha=.2)
plt.tight_layout(); plt.savefig(os.path.join(output_folder,"hierarchical_encoder_channels.png"),dpi=300,bbox_inches="tight"); plt.show()
display(hierarchical_channel_table.head()); hierarchical_channel_table.to_csv(os.path.join(output_folder,"hierarchical_encoder_channels.csv"),index=False)


## E. Effective branch share versus crime

In [ ]:
def association_stats_hier(x,y):
    x,y=np.asarray(x),np.asarray(y); valid=np.isfinite(x)&np.isfinite(y); x,y=x[valid],y[valid]
    if len(x)<3 or np.std(x)==0 or np.std(y)==0: return {"pearson_r":np.nan,"pearson_p":np.nan,"spearman_rho":np.nan,"spearman_p":np.nan}
    if SCIPY_AVAILABLE:
        pr,sr=pearsonr(x,y),spearmanr(x,y); return {"pearson_r":float(pr[0]),"pearson_p":float(pr[1]),"spearman_rho":float(sr[0]),"spearman_p":float(sr[1])}
    return {"pearson_r":float(np.corrcoef(x,y)[0,1]),"pearson_p":np.nan,"spearman_rho":np.nan,"spearman_p":np.nan}

def binned_hier(frame,x_col,y_col,bins=12):
    work=frame[[x_col,y_col]].replace([np.inf,-np.inf],np.nan).dropna().copy(); work["bin"]=pd.qcut(work[x_col],q=bins,duplicates="drop")
    out=work.groupby("bin",observed=True).agg(mean_x=(x_col,"mean"),mean_y=(y_col,"mean"),std_y=(y_col,"std"),count=(y_col,"size")).reset_index(drop=True); out["ci95"]=1.96*out.std_y/np.sqrt(out["count"].clip(lower=1)); return out

rows=[]; fig,axes=plt.subplots(1,2,figsize=(13,4.8))
for ax,x_col in zip(axes,["input_crime","target_crime"]):
    stats=association_stats_hier(hierarchical_frame[x_col],hierarchical_frame.static_effective_share); rows.append({"crime_variable":x_col,**stats}); summary=binned_hier(hierarchical_frame,x_col,"static_effective_share")
    ax.plot(summary.mean_x,summary.mean_y,marker="o"); ax.fill_between(summary.mean_x,summary.mean_y-summary.ci95,summary.mean_y+summary.ci95,alpha=.2)
    ax.set(title=f"Static occlusion share vs {x_col.replace('_',' ')}",xlabel=x_col.replace('_',' ').title(),ylabel="Static effective share",ylim=(0,1)); ax.legend([f"Spearman rho={stats['spearman_rho']:.3f}"]); ax.grid(alpha=.2)
plt.tight_layout(); plt.savefig(os.path.join(output_folder,"hierarchical_share_crime.png"),dpi=300,bbox_inches="tight"); plt.show()
hierarchical_crime_associations=pd.DataFrame(rows); display(hierarchical_crime_associations.round(5)); hierarchical_crime_associations.to_csv(os.path.join(output_folder,"hierarchical_share_crime.csv"),index=False)


## F. Spatial and temporal stability

In [ ]:
hierarchical_node_stability=hierarchical_frame.groupby("node_id").agg(static_share_mean=("static_effective_share","mean"),static_share_std=("static_effective_share","std"),mean_crime=("input_crime","mean"),mean_absolute_error=("absolute_error","mean")).reset_index()
hierarchical_time_stability=hierarchical_frame.groupby("time_id").agg(static_share_mean=("static_effective_share","mean"),static_share_std=("static_effective_share","std"),mean_crime=("input_crime","mean")).reset_index()
fig,axes=plt.subplots(1,3,figsize=(16,4.5)); sc=axes[0].scatter(hierarchical_node_stability.mean_crime,hierarchical_node_stability.static_share_mean,c=hierarchical_node_stability.mean_absolute_error,cmap="magma",s=18,alpha=.7); fig.colorbar(sc,ax=axes[0],label="Mean absolute error")
axes[0].set(title="Spatial effective branch balance",xlabel="Node mean crime",ylabel="Mean static share"); axes[1].hist(hierarchical_node_stability.static_share_std.dropna(),bins=30,color="tab:orange"); axes[1].set(title="Within-node temporal variability",xlabel="Static-share SD",ylabel="Nodes")
axes[2].plot(hierarchical_time_stability.time_id,hierarchical_time_stability.static_share_mean,marker="o"); axes[2].set(title="Citywide temporal static share",xlabel="Time ID",ylabel="Static share",ylim=(0,1))
for ax in axes: ax.grid(alpha=.2)
plt.tight_layout(); plt.savefig(os.path.join(output_folder,"hierarchical_spatial_temporal.png"),dpi=300,bbox_inches="tight"); plt.show()
hierarchical_node_stability.to_csv(os.path.join(output_folder,"hierarchical_node_stability.csv"),index=False); hierarchical_time_stability.to_csv(os.path.join(output_folder,"hierarchical_time_stability.csv"),index=False)


## G. Branch balance versus prediction error

In [ ]:
hierarchical_error_stats=association_stats_hier(hierarchical_frame.static_effective_share,hierarchical_frame.absolute_error); summary=binned_hier(hierarchical_frame,"static_effective_share","absolute_error",10)
plt.figure(figsize=(7,4.8)); plt.plot(summary.mean_x,summary.mean_y,marker="o"); plt.fill_between(summary.mean_x,summary.mean_y-summary.ci95,summary.mean_y+summary.ci95,alpha=.2)
plt.xlabel("Static effective share"); plt.ylabel("Mean absolute error"); plt.title("Hierarchical branch balance vs error"); plt.legend([f"Spearman rho={hierarchical_error_stats['spearman_rho']:.3f}"]); plt.grid(alpha=.2); plt.tight_layout(); plt.savefig(os.path.join(output_folder,"hierarchical_share_error.png"),dpi=300,bbox_inches="tight"); plt.show()
hierarchical_error_association=pd.DataFrame([hierarchical_error_stats]); display(hierarchical_error_association.round(5))


## H. Branch-removal and branch-reassignment faithfulness

In [ ]:
@torch.no_grad()
def evaluate_hierarchical_intervention(model,loader,device,mode="learned",random_seed=42):
    model.eval(); core=model.hierarchical_fusion; preds=[]; targets=[]
    for step,batch in enumerate(loader):
        batch=batch.to(device); zs=core.static_encoder(batch.x_static,batch.edge_index); zd=core.dynamic_encoder(batch.x_dynamic,batch.edge_index)
        if mode=="static_removed": zs=torch.zeros_like(zs)
        elif mode=="dynamic_removed": zd=torch.zeros_like(zd)
        elif mode=="both_removed": zs=torch.zeros_like(zs); zd=torch.zeros_like(zd)
        elif mode=="static_shuffled":
            g=torch.Generator(device=device); g.manual_seed(random_seed+step); zs=zs[torch.randperm(zs.shape[0],generator=g,device=device)]
        elif mode=="dynamic_shuffled":
            g=torch.Generator(device=device); g.manual_seed(random_seed+10000+step); zd=zd[torch.randperm(zd.shape[0],generator=g,device=device)]
        elif mode!="learned": raise ValueError(mode)
        prediction,_=core.forward_from_encoded(zs,zd,batch.edge_index); preds.append(prediction.cpu()); targets.append(batch.y.float().view_as(prediction).cpu())
    prediction=torch.cat(preds).numpy(); target=torch.cat(targets).numpy(); error=prediction-target; mse=float(np.mean(error**2)); return {"MSE":mse,"RMSE":float(np.sqrt(mse)),"MAE":float(np.mean(np.abs(error)))}

rows=[]
for label,mode in [("Learned hierarchy","learned"),("Static branch removed","static_removed"),("Dynamic branch removed","dynamic_removed"),("Both branches removed","both_removed"),("Static branch shuffled","static_shuffled"),("Dynamic branch shuffled","dynamic_shuffled")]: rows.append({"intervention":label,**evaluate_hierarchical_intervention(model,test_loader,device,mode,seed)})
hierarchical_branch_interventions=pd.DataFrame(rows); baseline=hierarchical_branch_interventions.loc[hierarchical_branch_interventions.intervention=="Learned hierarchy","MSE"].iloc[0]; hierarchical_branch_interventions["delta_MSE_vs_learned"]=hierarchical_branch_interventions.MSE-baseline; hierarchical_branch_interventions["relative_MSE_change_percent"]=100*hierarchical_branch_interventions.delta_MSE_vs_learned/(baseline+1e-12)
display(hierarchical_branch_interventions.round(6)); plot_df=hierarchical_branch_interventions.sort_values("delta_MSE_vs_learned")
plt.figure(figsize=(9,5)); plt.barh(plot_df.intervention,plot_df.delta_MSE_vs_learned,color=["tab:green" if x<=0 else "tab:red" for x in plot_df.delta_MSE_vs_learned]); plt.axvline(0,color="black"); plt.xlabel("Change in test MSE"); plt.title("Hierarchical branch-intervention faithfulness"); plt.grid(axis="x",alpha=.2); plt.tight_layout(); plt.savefig(os.path.join(output_folder,"hierarchical_branch_interventions.png"),dpi=300,bbox_inches="tight"); plt.show()
hierarchical_branch_interventions.to_csv(os.path.join(output_folder,"hierarchical_branch_interventions.csv"),index=False)


## I. Hierarchy-level intervention

Bypassing the joint GAT tests the additional hierarchical stage. It does not retrain the simpler architecture, so it is a frozen-model intervention. A separately trained late-fusion model remains the correct full baseline.

In [ ]:
@torch.no_grad()
def evaluate_hierarchy_level(model,loader,device,bypass=False,remove_edges=False):
    model.eval(); preds=[]; targets=[]
    for batch in loader:
        batch=batch.to(device); edges=batch.edge_index
        if remove_edges: edges=torch.empty((2,0),dtype=torch.long,device=device)
        prediction,_=model(batch.x_static,batch.x_dynamic,edges,bypass_joint_graph=bypass); preds.append(prediction.cpu()); targets.append(batch.y.float().view_as(prediction).cpu())
    prediction=torch.cat(preds).numpy(); target=torch.cat(targets).numpy(); error=prediction-target; mse=float(np.mean(error**2)); return {"MSE":mse,"RMSE":float(np.sqrt(mse)),"MAE":float(np.mean(np.abs(error)))}

hierarchy_level_interventions=pd.DataFrame([
    {"condition":"Complete hierarchy",**evaluate_hierarchy_level(model,test_loader,device,False,False)},
    {"condition":"Joint graph stage bypassed",**evaluate_hierarchy_level(model,test_loader,device,True,False)},
    {"condition":"All neighbor edges removed",**evaluate_hierarchy_level(model,test_loader,device,False,True)},
])
base=hierarchy_level_interventions.loc[hierarchy_level_interventions.condition=="Complete hierarchy","MSE"].iloc[0]; hierarchy_level_interventions["delta_MSE_vs_complete"]=hierarchy_level_interventions.MSE-base; hierarchy_level_interventions["relative_MSE_change_percent"]=100*hierarchy_level_interventions.delta_MSE_vs_complete/(base+1e-12)
display(hierarchy_level_interventions.round(6)); plt.figure(figsize=(8,4)); plt.bar(hierarchy_level_interventions.condition,hierarchy_level_interventions.MSE,color=["tab:blue","tab:purple","tab:gray"]); plt.xticks(rotation=12,ha="right"); plt.ylabel("Test MSE"); plt.title("Hierarchy and graph interventions"); plt.tight_layout(); plt.savefig(os.path.join(output_folder,"hierarchy_level_interventions.png"),dpi=300,bbox_inches="tight"); plt.show()
hierarchy_level_interventions.to_csv(os.path.join(output_folder,"hierarchy_level_interventions.csv"),index=False)


## J. Original-feature permutation importance

In [ ]:
@torch.no_grad()
def hierarchical_permutation_importance(model,loader,device,static_names=None,repeats=3,random_seed=42):
    model.eval(); baseline=hierarchical_metrics["MSE"]; sample=next(iter(loader)); static_dim=sample.x_static.shape[1]; dynamic_dim=sample.x_dynamic.shape[1]
    if static_names is None or len(static_names)!=static_dim: static_names=[f"static_{i}" for i in range(static_dim)]
    dynamic_names=[f"crime_lag_{dynamic_dim-i}" for i in range(dynamic_dim)]; rows=[]
    for modality,dim,names in [("static",static_dim,static_names),("dynamic",dynamic_dim,dynamic_names)]:
        for fi in range(dim):
            losses=[]
            for repeat in range(repeats):
                pred_all=[]; target_all=[]
                for step,batch in enumerate(loader):
                    batch=batch.to(device); xs=batch.x_static.clone(); xd=batch.x_dynamic.clone(); g=torch.Generator(device=device); g.manual_seed(random_seed+100000*(modality=="dynamic")+1000*fi+31*repeat+step); perm=torch.randperm(xs.shape[0],generator=g,device=device)
                    if modality=="static": xs[:,fi]=xs[perm,fi]
                    else: xd[:,fi]=xd[perm,fi]
                    prediction,_=model(xs,xd,batch.edge_index); pred_all.append(prediction.cpu()); target_all.append(batch.y.float().view_as(prediction).cpu())
                prediction=torch.cat(pred_all).numpy(); target=torch.cat(target_all).numpy(); losses.append(float(np.mean((prediction-target)**2)))
            rows.append({"modality":modality,"feature":names[fi],"baseline_MSE":baseline,"permuted_MSE_mean":np.mean(losses),"permuted_MSE_std":np.std(losses,ddof=1) if repeats>1 else 0,"delta_MSE":np.mean(losses)-baseline})
    return pd.DataFrame(rows)

static_feature_names=list(static_dt.columns) if len(static_dt.columns)==S else None
hierarchical_permutation_table=hierarchical_permutation_importance(model,test_loader,device,static_feature_names,3,seed); display(hierarchical_permutation_table.sort_values("delta_MSE",ascending=False).head(15)); plot_df=hierarchical_permutation_table.sort_values("delta_MSE")
plt.figure(figsize=(10,max(5,.32*len(plot_df)))); plt.barh(plot_df.feature,plot_df.delta_MSE,xerr=plot_df.permuted_MSE_std,color=plot_df.modality.map({"static":"tab:orange","dynamic":"tab:green"}),capsize=2); plt.axvline(0,color="black"); plt.xlabel("Increase in test MSE after permutation"); plt.title("Hierarchical input permutation importance"); plt.grid(axis="x",alpha=.2); plt.tight_layout(); plt.savefig(os.path.join(output_folder,"hierarchical_input_permutation.png"),dpi=300,bbox_inches="tight"); plt.show()
hierarchical_permutation_table.to_csv(os.path.join(output_folder,"hierarchical_input_permutation.csv"),index=False)


## K. Prediction diagnostics and exact-hop context

In [ ]:
pred_matrix=np.stack(hierarchical_details["predictions_per_time"]); target_matrix=np.stack(hierarchical_details["targets_per_time"]); y_pred=pred_matrix.reshape(-1); y_true=target_matrix.reshape(-1); residual=y_true-y_pred
fig,axes=plt.subplots(1,2,figsize=(11,4)); axes[0].scatter(y_true,y_pred,s=8,alpha=.3); lo=min(y_true.min(),y_pred.min()); hi=max(y_true.max(),y_pred.max()); axes[0].plot([lo,hi],[lo,hi],color="black",linestyle="--"); axes[0].set(title="Observed vs predicted",xlabel="Observed",ylabel="Predicted")
axes[1].scatter(y_pred,residual,s=8,alpha=.3); axes[1].axhline(0,color="black",linestyle="--"); axes[1].set(title="Residual plot",xlabel="Predicted",ylabel="Observed - predicted"); plt.tight_layout(); plt.savefig(os.path.join(output_folder,"hierarchical_prediction_diagnostics.png"),dpi=300,bbox_inches="tight"); plt.show()

def adjacency_hier(edge_index,num_nodes):
    adj={i:set() for i in range(num_nodes)}; src,dst=edge_index.detach().cpu().numpy()
    for u,v in zip(src,dst): adj[int(u)].add(int(v)); adj[int(v)].add(int(u))
    return adj
def exact_hop_hier(center,adj,hop):
    visited={center}; frontier={center}
    for _ in range(hop):
        nxt=set()
        for node in frontier: nxt.update(adj.get(node,set()))
        nxt-=visited; visited|=nxt; frontier=nxt
    return sorted(frontier)
adjacency=adjacency_hier(test_dataset[0].edge_index,pred_matrix.shape[1]); selected_node=int(np.argmax(target_matrix.mean(axis=0))); time=np.arange(pred_matrix.shape[0]); plt.figure(figsize=(9,5)); plt.plot(time,target_matrix[:,selected_node],color="black",linewidth=2.5,label="Target"); plt.plot(time,pred_matrix[:,selected_node],color="red",linewidth=2.2,label="Prediction")
for hop,color in zip([1,2],["tab:blue","tab:green"]):
    nodes=exact_hop_hier(selected_node,adjacency,hop)
    if nodes: plt.plot(time,pred_matrix[:,nodes].mean(axis=1),linestyle="--",color=color,label=f"Exact {hop}-hop mean (n={len(nodes)})")
plt.xlabel("Test time step"); plt.ylabel("Crime intensity"); plt.title(f"Hierarchical prediction context: node {selected_node}"); plt.legend(); plt.grid(alpha=.2); plt.tight_layout(); plt.savefig(os.path.join(output_folder,"hierarchical_high_crime_neighborhood.png"),dpi=300,bbox_inches="tight"); plt.show()


## L. Time-cluster bootstrap uncertainty

In [ ]:
def hierarchical_cluster_bootstrap(frame,n_bootstrap=1000,random_seed=42):
    rng=np.random.default_rng(random_seed); clusters=frame.time_id.drop_duplicates().to_numpy(); estimates=[]
    for _ in range(n_bootstrap):
        sampled=rng.choice(clusters,size=len(clusters),replace=True); boot=pd.concat([frame.loc[frame.time_id==c] for c in sampled],ignore_index=True); error=boot.prediction-boot.target_crime
        estimates.append({"MSE":np.mean(error**2),"MAE":np.mean(np.abs(error)),"static_effective_share":boot.static_effective_share.mean(),"share_crime_spearman":association_stats_hier(boot.input_crime,boot.static_effective_share)["spearman_rho"]})
    samples=pd.DataFrame(estimates); point={"MSE":hierarchical_metrics["MSE"],"MAE":hierarchical_metrics["MAE"],"static_effective_share":frame.static_effective_share.mean(),"share_crime_spearman":association_stats_hier(frame.input_crime,frame.static_effective_share)["spearman_rho"]}
    summary=pd.DataFrame([{"quantity":col,"estimate":point[col],"bootstrap_ci95_low":samples[col].quantile(.025),"bootstrap_ci95_high":samples[col].quantile(.975),"bootstrap_std":samples[col].std(ddof=1)} for col in samples.columns]); return summary,samples
hierarchical_bootstrap_summary,hierarchical_bootstrap_samples=hierarchical_cluster_bootstrap(hierarchical_frame,1000,seed); display(hierarchical_bootstrap_summary.round(6)); x=np.arange(len(hierarchical_bootstrap_summary)); lower=hierarchical_bootstrap_summary.estimate-hierarchical_bootstrap_summary.bootstrap_ci95_low; upper=hierarchical_bootstrap_summary.bootstrap_ci95_high-hierarchical_bootstrap_summary.estimate
plt.figure(figsize=(8,4.5)); plt.errorbar(x,hierarchical_bootstrap_summary.estimate,yerr=np.vstack([lower,upper]),fmt="o",capsize=5,linewidth=2); plt.xticks(x,hierarchical_bootstrap_summary.quantity,rotation=15,ha="right"); plt.ylabel("Estimate with time-cluster 95% CI"); plt.title("Hierarchical-fusion uncertainty"); plt.grid(axis="y",alpha=.2); plt.tight_layout(); plt.savefig(os.path.join(output_folder,"hierarchical_bootstrap_ci.png"),dpi=300,bbox_inches="tight"); plt.show(); hierarchical_bootstrap_summary.to_csv(os.path.join(output_folder,"hierarchical_bootstrap_ci.csv"),index=False)


## M. Hierarchy-stage embedding projections

In [ ]:
from sklearn.manifold import TSNE
n_first=len(hierarchical_details["targets_per_time"][0]); target_first=hierarchical_details["targets_per_time"][0]
stage_embeddings={"Pre-joint fused":hierarchical_details["h_fused"][:n_first],"Post-joint graph":hierarchical_details["h_joint"][:n_first]}
fig,axes=plt.subplots(1,2,figsize=(13,5.5))
for ax,(label,embedding) in zip(axes,stage_embeddings.items()):
    perplexity=min(30,max(2,(embedding.shape[0]-1)//3)); emb2d=TSNE(n_components=2,perplexity=perplexity,init="pca",learning_rate="auto",random_state=seed).fit_transform(embedding); scatter=ax.scatter(emb2d[:,0],emb2d[:,1],c=target_first,cmap="viridis",s=12,alpha=.8); ax.set(title=label,xlabel="t-SNE 1",ylabel="t-SNE 2"); pd.DataFrame({"tsne_1":emb2d[:,0],"tsne_2":emb2d[:,1],"target_crime":target_first}).to_csv(os.path.join(output_folder,f"hierarchical_tsne_{label.lower().replace(' ','_').replace('-','_')}.csv"),index=False)
fig.colorbar(scatter,ax=axes.ravel().tolist(),label="Future crime target"); fig.suptitle("Hierarchical representation stages"); fig.subplots_adjust(wspace=.25,top=.88,right=.9); plt.savefig(os.path.join(output_folder,"hierarchical_stage_tsne.png"),dpi=300,bbox_inches="tight"); plt.show()


## N. Exportable comparison summary

In [ ]:
static_removed=hierarchical_branch_interventions.query("intervention == 'Static branch removed'").iloc[0]; dynamic_removed=hierarchical_branch_interventions.query("intervention == 'Dynamic branch removed'").iloc[0]; static_shuffled=hierarchical_branch_interventions.query("intervention == 'Static branch shuffled'").iloc[0]; dynamic_shuffled=hierarchical_branch_interventions.query("intervention == 'Dynamic branch shuffled'").iloc[0]; bypassed=hierarchy_level_interventions.query("condition == 'Joint graph stage bypassed'").iloc[0]; edges_removed=hierarchy_level_interventions.query("condition == 'All neighbor edges removed'").iloc[0]; input_assoc=hierarchical_crime_associations.query("crime_variable == 'input_crime'").iloc[0]
hierarchical_model_summary=pd.DataFrame([{
    "model":"M8 hierarchical fusion","uses_graph":True,"fusion_location":"separate_graph_encoders_then_fusion_then_joint_graph","has_modality_gate":False,
    "test_MSE":hierarchical_metrics["MSE"],"test_RMSE":hierarchical_metrics["RMSE"],"test_MAE":hierarchical_metrics["MAE"],"test_R2":hierarchical_metrics["R2"],
    "static_only_MSE":dynamic_removed["MSE"],"dynamic_only_MSE":static_removed["MSE"],"static_gate_mean":np.nan,"gate_entropy":np.nan,
    "effective_static_proportion":hierarchical_frame.static_effective_share.mean(),"effective_share_input_crime_spearman":input_assoc["spearman_rho"],"mean_absolute_interaction":hierarchical_frame.interaction_output.abs().mean(),
    "static_branch_shuffle_delta_MSE":static_shuffled["MSE"]-hierarchical_metrics["MSE"],"dynamic_branch_shuffle_delta_MSE":dynamic_shuffled["MSE"]-hierarchical_metrics["MSE"],
    "joint_stage_bypass_delta_MSE":bypassed["MSE"]-hierarchical_metrics["MSE"],"edges_removed_delta_MSE":edges_removed["MSE"]-hierarchical_metrics["MSE"],"self_neighbor_analysis_available":False,
}]); display(hierarchical_model_summary.T); hierarchical_model_summary.to_csv(os.path.join(output_folder,"hierarchical_professional_summary.csv"),index=False)


## Interpretation framework

1. Compare forecasting metrics across identical preprocessing, splits, seeds and checkpoint rules.
2. Compare M8 directly with M7 to test whether post-fusion joint propagation adds value.
3. Do not describe occlusion shares as gates; the hierarchy has no probability-valued modality selector.
4. Report interaction magnitude because nonlinear hierarchical contributions need not add independently.
5. Use branch removal and shuffling together: removal tests availability, while shuffling tests case-specific assignment.
6. Treat the joint-stage bypass as a frozen-model diagnostic, not a replacement for the independently trained late-fusion baseline.
7. M11 alone supports explicit self-versus-neighbor modality gating.
8. Original GMU and GMU–GNN support feature-wise gates, while M7 and M8 support branch-level diagnostics rather than gate statistics.
9. Time-cluster bootstrap quantifies test-window variability; matched random seeds quantify training variability.